In [1]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import importlib

import sys

sys.path.append("/kaggle/input/models/leonidtikhanov/pinn-model/pytorch/default/1")

import pinn_model
pinn_model = importlib.reload(pinn_model)

print(pinn_model.__file__)
print("run_experiment:", hasattr(pinn_model, "run_experiment"))

/kaggle/input/models/leonidtikhanov/pinn-model/pytorch/default/1/pinn_model.py
run_experiment: True


In [2]:
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    device = "cuda"
    print("gpu:", torch.cuda.get_device_name(0))
else:
    device = "cpu"

print("device:", device)

torch version: 2.10.0+cu128
cuda available: True
gpu: Tesla T4
device: cuda


In [3]:
final_config = {
    "task_name": "heat1d",
    "dtype": "fp32",
    "seed": 0,
    "device": device,
    "alpha": 0.1,
    "hid_size": 64,
    "num_layers": 4,
    "n_collocation": 5000,
    "n_ic": 500,
    "n_bc": 500,
    "adam_steps": 8000,
    "lbfgs_steps": 500,
    "lr_adam": 1e-3,
    "use_adam": True,
    "use_lbfgs": True,
    "log_dir": "/kaggle/working/runs/heat1d_fp32_0",
}

In [4]:
all_summaries = []
all_histories = {}

for dtype in ["fp32", "fp64"]:
    for seed in [0, 1, 2]:
        config = final_config.copy()
        config["dtype"] = dtype
        config["seed"] = seed
        config["log_dir"] = f"/kaggle/working/runs/heat1d_{dtype}_{seed}"
        history, summary = pinn_model.run_experiment(config)
        all_summaries.append(summary)
        all_histories[f"{dtype}_seed{seed}"] = history

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


In [5]:
df = pd.DataFrame(all_summaries)
df.to_csv("results_summary.csv", index=False)
df

,task_name,dtype,seed,alpha,hid_size,num_layers,n_collocation,n_ic,n_bc,adam_steps,lbfgs_steps,final_loss,final_l2_error,time_sec,log_dir
0,heat1d,fp32,0,0.1,64,4,5000,500,500,8000,500,0.000002,0.000375,96.474978,/kaggle/working/runs/heat1d_fp32_0
1,heat1d,fp32,1,0.1,64,4,5000,500,500,8000,500,0.000001,0.000217,87.890680,/kaggle/working/runs/heat1d_fp32_1
2,heat1d,fp32,2,0.1,64,4,5000,500,500,8000,500,0.000008,0.000599,83.869530,/kaggle/working/runs/heat1d_fp32_2
3,heat1d,fp64,0,0.1,64,4,5000,500,500,8000,500,0.000007,0.000830,162.502528,/kaggle/working/runs/heat1d_fp64_0
4,heat1d,fp64,1,0.1,64,4,5000,500,500,8000,500,0.000003,0.000311,165.262592,/kaggle/working/runs/heat1d_fp64_1
5,heat1d,fp64,2,0.1,64,4,5000,500,500,8000,500,0.000002,0.000277,165.390579,/kaggle/working/runs/heat1d_fp64_2


In [6]:
grouped = df.groupby("dtype")[["final_l2_error", "final_loss", "time_sec"]].agg(["mean", "std"])
grouped.to_csv("results_grouped.csv")
grouped

final_l2_error           final_loss              time_sec          
                mean       std       mean       std        mean       std
dtype                                                                    
fp32        0.000397  0.000192   0.000004  0.000004   89.411730  6.438907
fp64        0.000473  0.000310   0.000004  0.000003  164.385233  1.631726